# Event-Level Merge and Preliminary Analysis

This notebook constructs an earnings-call event-level dataset by merging AI framing variables with daily turnover data. It uses an optimized trading-day-index approach:

- Standardize tickers across transcript and Bloomberg-style turnover data.
- Match each earnings call to the same ticker's event trading day 0, using the call date if available or the next trading day otherwise.
- Compute event and benchmark turnover windows from per-ticker trading-day indices and cumulative turnover sums.
- Estimate preliminary OLS regressions with firm-clustered standard errors.

In [ ]:
from pathlib import Path
import re
import warnings
import os

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp/codex_matplotlib_cache')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

warnings.filterwarnings('ignore', category=FutureWarning)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 140)

PROJECT_ROOT = Path.cwd().resolve()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_ROOT = PROJECT_ROOT / 'output'
OUTPUT_TABLES = OUTPUT_ROOT / 'tables'
OUTPUT_REGRESSIONS = OUTPUT_ROOT / 'regression_results'
OUTPUT_FIGURES = OUTPUT_ROOT / 'figures'

for directory in [OUTPUT_TABLES, OUTPUT_REGRESSIONS, OUTPUT_FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)

CALL_PATH = DATA_PROCESSED / 'earnings_call_ai_framing_dataset.csv'
TURNOVER_PATH = DATA_PROCESSED / 'turnover_daily_clean_2019_2025.csv'
EVENT_OUTPUT_PATH = DATA_PROCESSED / 'event_level_analysis_dataset.csv'

DESCRIPTIVE_PATH = OUTPUT_TABLES / 'descriptive_statistics.csv'
DESCRIPTIVE_BY_HYPE_PATH = OUTPUT_TABLES / 'descriptive_by_hype_period.csv'
OVERALL_REG_PATH = OUTPUT_REGRESSIONS / 'regression_table_overall_ai.txt'
FRAMING_REG_PATH = OUTPUT_REGRESSIONS / 'regression_table_ai_framing.txt'

HYPE_REFERENCE = '2019 Pre-Pandemic Baseline'
HYPE_ORDER = [
    '2019 Pre-Pandemic Baseline',
    '2021–2022 Pre-ChatGPT Transition',
    '2023 Initial GenAI Surge',
    '2024–2025 AI Adoption Period',
]
# Some environments normalize en dashes differently; this helper also supports hyphen variants.
HYPE_REFERENCE_FORMULA = 'C(hype_period, Treatment(reference="2019 Pre-Pandemic Baseline"))'

print(f'Project root: {PROJECT_ROOT}')
print('Setup complete.')

## 1. Load data

In [ ]:
calls_raw = pd.read_csv(CALL_PATH)
turnover_raw = pd.read_csv(TURNOVER_PATH, usecols=['ticker', 'trading_date', 'turnover'])

print('Data loaded')
print(f'Earnings-call AI framing rows: {len(calls_raw):,}; columns: {len(calls_raw.columns):,}')
print(f'Daily turnover rows: {len(turnover_raw):,}; columns: {len(turnover_raw.columns):,}')
print('\nCall-level sample:')
display(calls_raw.head(3))
print('\nTurnover sample:')
display(turnover_raw.head(3))

## 2. Standardize tickers and convert dates

Bloomberg tickers such as `AAPL US Equity` are standardized to the common ticker stem `AAPL`. The same normalization is applied to transcript tickers.

In [ ]:
def standardize_ticker(value):
    """Return a compact ticker key suitable for matching transcript and Bloomberg-style tickers."""
    if pd.isna(value):
        return pd.NA
    text = str(value).upper().strip()
    text = re.sub(r'\s+', ' ', text)
    # Bloomberg format commonly appears as "AAPL US Equity"; keep the security stem.
    text = re.sub(r'\s+(?:US|UN|UW|UR|LN|JP|HK|CN|KS|KQ|GY|GR|FP|SW|SS|SZ|AU|CA|CT|TO|PA|MI|IM|SM|NA|BB|BR|NO|DC|FH|SS|SE)\s+EQUITY$', '', text)
    text = re.sub(r'\s+EQUITY$', '', text)
    # If any residual Bloomberg suffix remains, use the first token as the common ticker stem.
    if ' ' in text:
        text = text.split(' ')[0]
    text = text.replace('.', '').replace('/', '').replace('-', '')
    return text if text else pd.NA

calls = calls_raw.copy()
turnover = turnover_raw.copy()

calls['ticker_original'] = calls['ticker']
turnover['ticker_original'] = turnover['ticker']

calls['ticker_standardized'] = calls['ticker'].map(standardize_ticker)
turnover['ticker_standardized'] = turnover['ticker'].map(standardize_ticker)

print('Tickers standardized')
print(f"Unique standardized call tickers: {calls['ticker_standardized'].nunique():,}")
print(f"Unique standardized turnover tickers: {turnover['ticker_standardized'].nunique():,}")
print('\nExample turnover ticker standardization:')
display(turnover[['ticker_original', 'ticker_standardized']].drop_duplicates().head(10))

calls['call_date'] = pd.to_datetime(calls['call_date'], errors='coerce')
turnover['trading_date'] = pd.to_datetime(turnover['trading_date'], errors='coerce')

print('Dates converted')
print(f"Call date range: {calls['call_date'].min().date()} to {calls['call_date'].max().date()}")
print(f"Trading date range: {turnover['trading_date'].min().date()} to {turnover['trading_date'].max().date()}")

## 3. Exclude 2020 earnings calls from main analysis sample

The main analysis excludes 2020 because COVID-19 created abnormal market conditions.

In [ ]:
pre_exclusion_count = len(calls)
mask_2020 = calls['call_date'].dt.year == 2020
removed_2020 = int(mask_2020.sum())
calls_main = calls.loc[~mask_2020].copy()

# Keep hype-period ordering stable for descriptives and regressions.
if 'hype_period' in calls_main.columns:
    observed_hype = [x for x in HYPE_ORDER if x in set(calls_main['hype_period'].dropna())]
    remaining_hype = [x for x in calls_main['hype_period'].dropna().unique() if x not in observed_hype]
    calls_main['hype_period'] = pd.Categorical(calls_main['hype_period'], categories=observed_hype + remaining_hype, ordered=True)

print('2020 observations checked/removed')
print(f'Number of earnings calls before merge: {pre_exclusion_count:,}')
print(f'Number of 2020 observations removed: {removed_2020:,}')
print(f'Number of earnings calls after excluding 2020: {len(calls_main):,}')
print('\nHype-period counts after 2020 exclusion:')
print(calls_main['hype_period'].value_counts(dropna=False).sort_index())

## 4. Create trading-day index and cumulative turnover sums

Each ticker's turnover rows are sorted by trading date and assigned a dense trading-day index. Cumulative sums allow fast window averages without repeatedly scanning the full turnover dataframe.

In [ ]:
turnover = turnover.dropna(subset=['ticker_standardized', 'trading_date', 'turnover']).copy()
turnover['turnover'] = pd.to_numeric(turnover['turnover'], errors='coerce')
turnover = turnover.dropna(subset=['turnover'])
turnover = turnover.sort_values(['ticker_standardized', 'trading_date'])
turnover = turnover.drop_duplicates(subset=['ticker_standardized', 'trading_date'], keep='first')
turnover['trading_day_index'] = turnover.groupby('ticker_standardized').cumcount().astype(int)
turnover['cum_turnover'] = turnover.groupby('ticker_standardized')['turnover'].cumsum()

print('Trading-day index created')
print(f'Turnover rows after cleaning/deduplication: {len(turnover):,}')
print(f"Ticker groups with turnover data: {turnover['ticker_standardized'].nunique():,}")
print('\nTrading-day index sample:')
display(turnover.head(5))

## 5. Match each call date to event trading day 0

For each ticker, the event day is the same trading date if the call date is a trading day; otherwise it is the next available trading day.

In [ ]:
# Build compact per-ticker arrays once, then use searchsorted within ticker groups.
turnover_arrays = {
    ticker: {
        'dates': grp['trading_date'].to_numpy(dtype='datetime64[ns]'),
        'indices': grp['trading_day_index'].to_numpy(dtype=int),
    }
    for ticker, grp in turnover.groupby('ticker_standardized', sort=False)
}

calls_main = calls_main.reset_index(drop=True)
calls_main['_call_row_id'] = np.arange(len(calls_main))
calls_main['event_trading_date'] = pd.NaT
calls_main['event_trading_day_index'] = np.nan
calls_main['has_matching_turnover_ticker'] = calls_main['ticker_standardized'].isin(turnover_arrays.keys())

matched_chunks = []
for ticker, grp in calls_main.groupby('ticker_standardized', dropna=False, sort=False):
    if pd.isna(ticker) or ticker not in turnover_arrays:
        continue
    date_array = turnover_arrays[ticker]['dates']
    index_array = turnover_arrays[ticker]['indices']
    call_dates = grp['call_date'].to_numpy(dtype='datetime64[ns]')
    positions = np.searchsorted(date_array, call_dates, side='left')
    valid = positions < len(date_array)
    if not valid.any():
        continue
    chunk = pd.DataFrame({
        '_call_row_id': grp['_call_row_id'].to_numpy()[valid],
        'event_trading_date': pd.to_datetime(date_array[positions[valid]]),
        'event_trading_day_index': index_array[positions[valid]],
    })
    matched_chunks.append(chunk)

if matched_chunks:
    event_matches = pd.concat(matched_chunks, ignore_index=True)
    calls_main = calls_main.drop(columns=['event_trading_date', 'event_trading_day_index']).merge(event_matches, on='_call_row_id', how='left')
else:
    calls_main['event_trading_date'] = pd.NaT
    calls_main['event_trading_day_index'] = np.nan

calls_main['event_match_lag_calendar_days'] = (calls_main['event_trading_date'] - calls_main['call_date']).dt.days

# The next trading day should normally be the same day or within a short weekend/holiday gap.
# Very large lags indicate missing turnover coverage or a stale ticker match, so exclude them.
MAX_EVENT_MATCH_LAG_DAYS = 7
stale_match_mask = calls_main['event_match_lag_calendar_days'] > MAX_EVENT_MATCH_LAG_DAYS
stale_match_count = int(stale_match_mask.sum())
calls_main.loc[stale_match_mask, ['event_trading_date', 'event_trading_day_index', 'event_match_lag_calendar_days']] = [pd.NaT, np.nan, np.nan]

calls_with_matching_turnover = int(calls_main['event_trading_day_index'].notna().sum())

print('Call dates matched to trading-day index')
print(f'Number of stale matches removed because lag > {MAX_EVENT_MATCH_LAG_DAYS} calendar days: {stale_match_count:,}')
print(f'Number of calls with matching turnover data: {calls_with_matching_turnover:,}')
print('Calendar-day lag from call_date to event trading date:')
print(calls_main['event_match_lag_calendar_days'].describe())
print('\nLag distribution:')
print(calls_main['event_match_lag_calendar_days'].value_counts(dropna=False).sort_index().head(10))

## 6. Calculate event and benchmark turnover windows efficiently

Windows are based on trading-day indices:

- Event window `[0,+1]`: indices `t` to `t+1`
- Benchmark window `[-60,-11]`: indices `t-60` to `t-11`

In [ ]:
# Helper lookup table for cumulative turnover at specific ticker-index pairs.
cum_lookup = turnover[['ticker_standardized', 'trading_day_index', 'cum_turnover']].copy()
cum_lookup = cum_lookup.rename(columns={'trading_day_index': '_lookup_index', 'cum_turnover': '_lookup_cum_turnover'})

# Ticker-level metadata gives max index and supports window boundary diagnostics.
ticker_max_index = turnover.groupby('ticker_standardized')['trading_day_index'].max().rename('max_trading_day_index')
events = calls_main.merge(ticker_max_index, on='ticker_standardized', how='left')

events['event_start_index'] = events['event_trading_day_index']
events['event_end_index'] = events['event_trading_day_index'] + 1
events['benchmark_start_index'] = events['event_trading_day_index'] - 60
events['benchmark_end_index'] = events['event_trading_day_index'] - 11

# A valid event window requires both t and t+1. A valid benchmark requires t-60 through t-11.
events['valid_event_window_bounds'] = (
    events['event_trading_day_index'].notna()
    & (events['event_end_index'] <= events['max_trading_day_index'])
)
events['valid_benchmark_window_bounds'] = (
    events['event_trading_day_index'].notna()
    & (events['benchmark_start_index'] >= 0)
)

# Merge cumulative sums for event end, event start - 1, benchmark end, and benchmark start - 1.
def attach_cum_at_index(df, index_col, output_col):
    temp = df[['_call_row_id', 'ticker_standardized', index_col]].copy()
    temp = temp.rename(columns={index_col: '_lookup_index'})
    temp['_lookup_index'] = temp['_lookup_index'].astype('float')
    valid = temp['_lookup_index'].notna()
    temp.loc[valid, '_lookup_index'] = temp.loc[valid, '_lookup_index'].astype(int)
    merged = temp.merge(cum_lookup, on=['ticker_standardized', '_lookup_index'], how='left')
    return merged.set_index('_call_row_id')['_lookup_cum_turnover'].rename(output_col)

events['event_start_minus_one_index'] = events['event_start_index'] - 1
events['benchmark_start_minus_one_index'] = events['benchmark_start_index'] - 1

for index_col, output_col in [
    ('event_end_index', 'event_end_cum'),
    ('event_start_minus_one_index', 'event_start_minus_one_cum'),
    ('benchmark_end_index', 'benchmark_end_cum'),
    ('benchmark_start_minus_one_index', 'benchmark_start_minus_one_cum'),
]:
    events = events.merge(attach_cum_at_index(events, index_col, output_col), left_on='_call_row_id', right_index=True, how='left')

# When the start index is 0, the cumulative sum before the window is 0.
events.loc[events['event_start_minus_one_index'] < 0, 'event_start_minus_one_cum'] = 0.0
events.loc[events['benchmark_start_minus_one_index'] < 0, 'benchmark_start_minus_one_cum'] = 0.0

events['event_turnover_sum'] = events['event_end_cum'] - events['event_start_minus_one_cum']
events['benchmark_turnover_sum'] = events['benchmark_end_cum'] - events['benchmark_start_minus_one_cum']

events['avg_turnover_event'] = np.where(events['valid_event_window_bounds'], events['event_turnover_sum'] / 2, np.nan)
events['avg_turnover_benchmark'] = np.where(events['valid_benchmark_window_bounds'], events['benchmark_turnover_sum'] / 50, np.nan)
events['abnormal_turnover_diff'] = events['avg_turnover_event'] - events['avg_turnover_benchmark']
events['log_abnormal_turnover'] = np.where(
    (events['avg_turnover_event'] > 0) & (events['avg_turnover_benchmark'] > 0),
    np.log(events['avg_turnover_event']) - np.log(events['avg_turnover_benchmark']),
    np.nan,
)

valid_event = int(events['avg_turnover_event'].notna().sum())
valid_benchmark = int(events['avg_turnover_benchmark'].notna().sum())
valid_log_abn = int(events['log_abnormal_turnover'].notna().sum())

print('Event windows calculated')
print(f'Number of calls with valid [0,+1] event-window turnover: {valid_event:,}')
print(f'Number of calls with valid [-60,-11] benchmark-window turnover: {valid_benchmark:,}')
print(f'Number of final observations with valid log_abnormal_turnover: {valid_log_abn:,}')

## 7. Save final event-level analysis dataset

In [ ]:
helper_cols = [
    '_call_row_id', 'event_start_index', 'event_end_index', 'benchmark_start_index', 'benchmark_end_index',
    'event_start_minus_one_index', 'benchmark_start_minus_one_index', 'event_end_cum', 'event_start_minus_one_cum',
    'benchmark_end_cum', 'benchmark_start_minus_one_cum', 'event_turnover_sum', 'benchmark_turnover_sum'
]
export_cols = [c for c in events.columns if c not in helper_cols]
event_level = events[export_cols].copy()
event_level.to_csv(EVENT_OUTPUT_PATH, index=False)

print('Final sample size reported')
print(f'Saved event-level analysis dataset: {EVENT_OUTPUT_PATH}')
print(f'Rows: {len(event_level):,}; columns: {len(event_level.columns):,}')
print(f"Valid log_abnormal_turnover rows: {event_level['log_abnormal_turnover'].notna().sum():,}")
display(event_level.head(5))

## 8. Descriptive statistics

In [ ]:
analysis_vars = [
    'ai_intensity', 'opportunity_intensity', 'implementation_intensity', 'risk_intensity',
    'opportunity_share', 'implementation_share', 'risk_share', 'mixed_unclear_share',
    'avg_turnover_event', 'avg_turnover_benchmark', 'abnormal_turnover_diff', 'log_abnormal_turnover',
]
analysis_vars = [v for v in analysis_vars if v in event_level.columns]

desc = event_level[analysis_vars].describe(percentiles=[0.25, 0.5, 0.75]).T
desc['missing'] = event_level[analysis_vars].isna().sum()
desc.to_csv(DESCRIPTIVE_PATH)

by_hype_metrics = [
    'ai_intensity', 'opportunity_intensity', 'implementation_intensity', 'risk_intensity',
    'log_abnormal_turnover', 'abnormal_turnover_diff', 'avg_turnover_event', 'avg_turnover_benchmark'
]
by_hype_metrics = [v for v in by_hype_metrics if v in event_level.columns]

desc_by_hype = (
    event_level
    .groupby('hype_period', observed=True)[by_hype_metrics]
    .agg(['count', 'mean', 'std', 'median'])
)
desc_by_hype.to_csv(DESCRIPTIVE_BY_HYPE_PATH)

print('Descriptive statistics saved')
print(f'- {DESCRIPTIVE_PATH}')
print(f'- {DESCRIPTIVE_BY_HYPE_PATH}')
print('\nOverall descriptive statistics:')
display(desc)
print('\nSelected means by hype period:')
display(event_level.groupby('hype_period', observed=True)[['ai_intensity', 'opportunity_intensity', 'implementation_intensity', 'risk_intensity', 'log_abnormal_turnover']].mean())

## 9. OLS regressions with firm-clustered standard errors

The dependent variable is `log_abnormal_turnover`. Standard errors are clustered by `ticker_standardized`. The 2019 pre-pandemic period is the reference category for hype-period indicators.

In [ ]:
regression_vars = [
    'log_abnormal_turnover', 'ai_intensity', 'opportunity_intensity', 'implementation_intensity',
    'risk_intensity', 'hype_period', 'ticker_standardized'
]
reg_df = event_level.dropna(subset=regression_vars).copy()
reg_df = reg_df[reg_df['ticker_standardized'].notna()].copy()

# Ensure the reference category is available and ordered for Patsy treatment coding.
reg_df['hype_period'] = reg_df['hype_period'].astype(str)
if HYPE_REFERENCE not in set(reg_df['hype_period']):
    raise ValueError(f'Reference hype period not found in regression sample: {HYPE_REFERENCE}')

hype_term = 'C(hype_period, Treatment(reference="2019 Pre-Pandemic Baseline"))'
ticker_fe = 'C(ticker_standardized)'

formulas_overall = {
    'Model 1': 'log_abnormal_turnover ~ ai_intensity',
    'Model 2': f'log_abnormal_turnover ~ ai_intensity + {hype_term}',
    'Model 3': f'log_abnormal_turnover ~ ai_intensity + {hype_term} + {ticker_fe}',
}

formulas_framing = {
    'Model 4': f'log_abnormal_turnover ~ opportunity_intensity + implementation_intensity + risk_intensity + {hype_term} + {ticker_fe}',
    'Model 5': f'log_abnormal_turnover ~ opportunity_intensity * {hype_term} + implementation_intensity * {hype_term} + risk_intensity * {hype_term} + {ticker_fe}',
}


def fit_clustered(formula, data):
    model = smf.ols(formula=formula, data=data)
    result = model.fit(cov_type='cluster', cov_kwds={'groups': data['ticker_standardized']})
    return result

print('Regression sample prepared')
print(f'Regression observations: {len(reg_df):,}')
print(f"Regression ticker clusters: {reg_df['ticker_standardized'].nunique():,}")

models_overall = {name: fit_clustered(formula, reg_df) for name, formula in formulas_overall.items()}
models_framing = {name: fit_clustered(formula, reg_df) for name, formula in formulas_framing.items()}

info_dict = {
    'N': lambda x: f'{int(x.nobs):,}',
    'R2': lambda x: f'{x.rsquared:.3f}',
    'Adj. R2': lambda x: f'{x.rsquared_adj:.3f}',
}

overall_table = summary_col(
    list(models_overall.values()),
    model_names=list(models_overall.keys()),
    stars=True,
    float_format='%0.4f',
    info_dict=info_dict,
)
framing_table = summary_col(
    list(models_framing.values()),
    model_names=list(models_framing.keys()),
    stars=True,
    float_format='%0.4f',
    info_dict=info_dict,
)

with open(OVERALL_REG_PATH, 'w') as f:
    f.write('Table 1: Overall AI intensity models\n')
    f.write('Dependent variable: log_abnormal_turnover\n')
    f.write('Standard errors clustered by ticker_standardized.\n\n')
    f.write(overall_table.as_text())

with open(FRAMING_REG_PATH, 'w') as f:
    f.write('Table 2: AI framing models\n')
    f.write('Dependent variable: log_abnormal_turnover\n')
    f.write('Standard errors clustered by ticker_standardized.\n\n')
    f.write(framing_table.as_text())

print('Regressions completed')
print(f'Saved overall AI regression table: {OVERALL_REG_PATH}')
print(f'Saved AI framing regression table: {FRAMING_REG_PATH}')
print('\nTable 1 preview:')
print(overall_table.as_text()[:2500])
print('\nTable 2 preview:')
print(framing_table.as_text()[:2500])

## 10. Presentation-ready figures

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

# Figure 1: Average AI intensity by hype period.
fig1_data = event_level.groupby('hype_period', observed=True)['ai_intensity'].mean().reindex([x for x in HYPE_ORDER if x in set(event_level['hype_period'].astype(str))])
fig, ax = plt.subplots(figsize=(10, 5.5))
fig1_data.plot(kind='bar', ax=ax, color='#2F6F73')
ax.set_title('Average AI Intensity by Hype Period', fontsize=14, weight='bold')
ax.set_xlabel('')
ax.set_ylabel('Mean AI intensity')
ax.tick_params(axis='x', rotation=30)
ax.set_ylim(bottom=0)
fig.tight_layout()
fig1_path = OUTPUT_FIGURES / 'figure_1_average_ai_intensity_by_hype_period.png'
fig.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.close(fig)

# Figure 2: Opportunity / implementation / risk shares by hype period.
share_cols = ['opportunity_share', 'implementation_share', 'risk_share']
fig2_data = event_level.groupby('hype_period', observed=True)[share_cols].mean().reindex(fig1_data.index)
fig, ax = plt.subplots(figsize=(10, 5.5))
fig2_data.plot(kind='bar', ax=ax, color=['#3B7A57', '#3A5FCD', '#A23E48'])
ax.set_title('Average AI Framing Shares by Hype Period', fontsize=14, weight='bold')
ax.set_xlabel('')
ax.set_ylabel('Mean share of AI-related sentences')
ax.tick_params(axis='x', rotation=30)
ax.legend(['Opportunity', 'Implementation', 'Risk'], frameon=False)
ax.set_ylim(bottom=0)
fig.tight_layout()
fig2_path = OUTPUT_FIGURES / 'figure_2_ai_framing_shares_by_hype_period.png'
fig.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.close(fig)

# Figure 3: Average log abnormal turnover by AI intensity group.
fig3_df = event_level[event_level['log_abnormal_turnover'].notna()].copy()
fig3_df['ai_intensity_group'] = np.where(fig3_df['ai_intensity'] > 0, 'AI-related calls', 'No AI-related sentences')
fig3_data = fig3_df.groupby('ai_intensity_group')['log_abnormal_turnover'].agg(['mean', 'count', 'std'])
fig3_data['se'] = fig3_data['std'] / np.sqrt(fig3_data['count'])
fig3_data = fig3_data.reindex(['No AI-related sentences', 'AI-related calls'])

fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.bar(fig3_data.index, fig3_data['mean'], yerr=1.96 * fig3_data['se'], capsize=5, color=['#7A7A7A', '#2F6F73'])
ax.axhline(0, color='black', linewidth=0.9)
ax.set_title('Average Log Abnormal Turnover by AI Intensity Group', fontsize=14, weight='bold')
ax.set_xlabel('')
ax.set_ylabel('Mean log abnormal turnover')
fig.tight_layout()
fig3_path = OUTPUT_FIGURES / 'figure_3_log_abnormal_turnover_by_ai_group.png'
fig.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.close(fig)

print('Figures saved')
print(f'- {fig1_path}')
print(f'- {fig2_path}')
print(f'- {fig3_path}')